### Day 9 Assignment: Databricks CLI, Declarative Automation Bundles & GitHub Actions

### Basic Tasks 


#### 1. Install and authenticate the Databricks CLI 

##### Objective
Install the Databricks CLI and authenticate with the Databricks workspace using OAuth U2M.

##### Commands Used

```bash
databricks --version
databricks auth login
databricks current-user me

![image_1788936596178.png](./image_1788936596178.png "image_1788936596178.png")

#### 2. Initialize a Declarative Automation Bundle project

Created a new **Declarative Automation Bundle (DAB)** project named `Assignment_9_Dabs`.

The bundle contains:
- `databricks.yml` for project and target configuration.
- A job resource under `resources/`.
- Source notebook/code for the job


![Screenshot 2026-09-09 130238_1788939255878.png](./Screenshot 2026-09-09 130238_1788939255878.png "Screenshot 2026-09-09 130238_1788939255878.png")

      


#### 3. Validate and Deploy to Dev
Ran the following commands:

```bash
databricks bundle validate -t dev
databricks bundle deploy -t dev

![Screenshot 2026-09-09 131351_1788940161455.png](./Screenshot 2026-09-09 131351_1788940161455.png "Screenshot 2026-09-09 131351_1788940161455.png")

### Intermediate Tasks

#### 4. run_as Service Principal

A second target named **`prod`** was configured in `databricks.yml` to represent the production environment.

The `dev` target points to the development workspace, while the `prod` target points to a separate production workspace.

The production target is configured with a **Service Principal** using `run_as`, so jobs deployed to production run using the service principal identity rather than an individual user's identity.

**Authentication**

U2M authentication was used to access the production workspace during development. The Service Principal configured under run_as is the identity intended for the production job execution.

![image_1789035636607.png](./image_1789035636607.png "image_1789035636607.png")

#### 5. Configure M2M Service Principal Authentication
Step 1: Create a Service Principal

In Databricks:

`Settings → Identity and access → Service principals`

Create a service principal, for example:

`cyntexa-prod-sp`

Enable **Workspace access** and assign the required permissions.

Step 2: Get Application ID

Open the service principal and copy its:

`Application ID`

This is used as the OAuth `client_id`.

Step 3: Create OAuth Secret

Open the service principal's **OAuth secrets** section and create a new secret.

Save the generated secret securely because it is used as the:

`client_secret`

Step 4: Configure Databricks CLI

Set the following environment variables:

```bash
export DATABRICKS_HOST="https://<prod-workspace-url>"
export DATABRICKS_CLIENT_ID="<application-id>"
export DATABRICKS_CLIENT_SECRET="<oauth-secret>"

#in .databrickscfg
[m2m]
host = "https://<prod-workspace-url>"
client_id = "<application-id>"
client_secret = "<oauth-secret>"
``` 
Step 5: Test Authentication

Run:

```bash
databricks current-user me
```

The returned identity should be the service principal rather than a personal user.

Step 6: Validate the Bundle and Deploy

![image_1789111834211.png](./image_1789111834211.png "image_1789111834211.png")

#### 6. GitHub Actions: Validate DAB

Configured a GitHub Actions workflow to validate the Databricks Asset Bundle whenever a Pull Request is created or updated against the `main` branch.

The workflow:

1. Checks out the repository.
2. Installs the Databricks CLI.
3. Provides Databricks authentication details through GitHub Actions Secrets.
4. Runs `databricks bundle validate -t prod` from the DAB project directory.
5. Does **not** deploy or modify any Databricks resources.

For authentication using the Databricks Service Principal, the following values are stored securely as GitHub Actions **Secrets**:

- `DATABRICKS_HOST` — Databricks workspace URL
- `DATABRICKS_CLIENT_ID` — Service Principal Application ID
- `DATABRICKS_CLIENT_SECRET` — Service Principal OAuth secret

The client secret is never hard-coded in the workflow or source code.

```yaml
name: Validate DAB

on:
  pull_request:
    branches:
      - main

jobs:
  validate:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate Bundle
        working-directory: Practice_Assignments/Day9/Assignment_9_Dabs
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
          DATABRICKS_AUTH_TYPE: oauth-m2m
        run: databricks bundle validate -t prod

###  Advanced Tasks 

#### 7. GitHub Actions: Deploy DAB to Production using OIDC

Extended the GitHub Actions workflow to deploy the Databricks Asset Bundle to the production environment after changes are merged into the `main` branch.

OIDC (OpenID Connect) authentication is used so that no Databricks client secret needs to be stored in GitHub. GitHub generates a short-lived identity token, which is used to authenticate the configured Databricks Service Principal.

The workflow:

1. Runs when changes are pushed to the `main` branch.
2. Checks out the repository.
3. Installs the Databricks CLI.
4. Requests an OIDC identity token using `id-token: write`.
5. Authenticates to Databricks using the configured Service Principal.
6. Validates the production bundle.
7. Deploys the validated bundle to the production target.

```yaml
name: Deploy DAB to Production

on:
  push:
    branches:
      - main

permissions:
  id-token: write
  contents: read

jobs:
  deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate Bundle
        working-directory: Practice_Assignments/Day9/Assignment_9_Dabs
        env:
          DATABRICKS_AUTH_TYPE: github-oidc
          DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
        run: databricks bundle validate -t prod

      - name: Deploy Bundle
        working-directory: Practice_Assignments/Day9/Assignment_9_Dabs
        env:
          DATABRICKS_AUTH_TYPE: github-oidc
          DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
        run: databricks bundle deploy -t prod

#### 8. Rollback Plan

Designed a rollback strategy for production in case a Databricks Asset Bundle deployment introduces an issue with a production job or resource.

The rollback approach is based on **Git version control**, where the previously working version of the bundle can be identified and redeployed.

The rollback process:

1. Identify the last known working Git commit.
2. Check out the previous working version.
3. Validate the bundle using the production target.
4. Redeploy the validated previous version to production.
5. Verify that the affected job/resource is working correctly.

First, identify the previous working commit:

```bash
git log --oneline
```

Check out the known working version:

```bash
git checkout <previous-working-commit>
```

Validate the previous bundle:

```bash
databricks bundle validate -t prod
```

If validation succeeds, redeploy it:

```bash
databricks bundle deploy -t prod
```

After confirming that production is stable, return to the main branch:

```bash
git checkout main
```

This approach provides a quick and reproducible rollback because the complete bundle configuration is maintained in Git. Instead of manually changing individual production resources, the last known working bundle version can be redeployed.

#### 9. Production Onboarding Guide

A Databricks Asset Bundle provides a structured way to manage Databricks resources through configuration files, Git, the Databricks CLI, and GitHub Actions.

A typical change follows this lifecycle:

```text
Local Development
        ↓
Edit databricks.yml / Resource Files
        ↓
Local Bundle Validation
        ↓
Git Commit & Push
        ↓
Pull Request
        ↓
GitHub Actions Validation
        ↓
Code Review
        ↓
Merge to main
        ↓
Production Deployment
```

### 1. Make Changes Locally

A developer makes the required changes to the DAB project, such as modifying:

```text
databricks.yml
resources/
src/
```

The `databricks.yml` file defines the bundle configuration and deployment targets such as `dev` and `prod`.

### 2. Validate the Bundle

Before committing the changes, validate the bundle locally:

```bash
databricks bundle validate -t dev
```

This checks the bundle configuration before deployment.

### 3. Commit and Push

Once the local validation succeeds:

```bash
git add .
git commit -m "Update Databricks bundle"
git push origin <feature-branch>
```

### 4. Create a Pull Request

Create a Pull Request from the feature branch to `main`.

GitHub Actions automatically runs the validation workflow:

```bash
databricks bundle validate -t prod
```

The workflow checks whether the production bundle configuration is valid without deploying it.

### 5. Review and Merge

The changes are reviewed by the team. Once the Pull Request is approved, it is merged into `main`.

### 6. Deploy to Production

The merge to `main` triggers the production deployment workflow.

GitHub Actions:

1. Checks out the repository.
2. Installs the Databricks CLI.
3. Authenticates with the Databricks workspace.
4. Validates the production bundle.
5. Deploys the bundle using:

```bash
databricks bundle deploy -t prod
```

The `prod` target in `databricks.yml` determines the production workspace and deployment configuration.

### 7. Rollback if Required

If a production deployment causes an issue, identify the last known working Git version and redeploy it:

```bash
git checkout <previous-working-commit>
databricks bundle validate -t prod
databricks bundle deploy -t prod
```

### Key Commands

```bash
# Validate development
databricks bundle validate -t dev

# Validate production configuration
databricks bundle validate -t prod

# Deploy to production
databricks bundle deploy -t prod

# View Git history for rollback
git log --oneline
```

### Safe Production Flow

```text
Edit
 → Validate locally
 → Commit
 → Push feature branch
 → Pull Request
 → CI validation
 → Code review
 → Merge to main
 → CI/CD production deployment
 → Monitor
 → Rollback if required
```

This workflow ensures that production changes are version-controlled, validated, reviewed, and deployed through an automated CI/CD process.